# NSI Ingest + Analysis-Ready Export Demo

End-to-end walk-through of the RQM NSI-minimal workflow:

1. Boot a local Apache Sedona session over Apache Iceberg storage.
2. Load `test/data/nsi.parquet` into `rqm.buildings` and `rqm.geometries`.
3. Validate the ingest against the ADR invariants.
4. Run a handful of example analytical queries.
5. Export a **flat, analysis-ready** table joining buildings with their
   computation-selected geometry, including WGS84 lat/lon and the fields a
   downstream flood-loss engine typically consumes.

**Prerequisites.** Run inside the devcontainer (`rqm-base` env) or install
`tools/requirements.txt` with Java 17 on `PATH`.

## 0. Setup

In [1]:
import sys
from pathlib import Path

# Locate the repo root by walking up from the notebook's cwd.
REPO = Path.cwd().resolve()
while (
    REPO != REPO.parent
    and not (REPO / "data-dictionary" / "data-dictionary.yaml").exists()
):
    REPO = REPO.parent
assert (REPO / "data-dictionary" / "data-dictionary.yaml").exists(), (
    f"could not locate repo root from {Path.cwd()}"
)

sys.path.insert(0, str(REPO / "tools"))

PARQUET = REPO / "test" / "data" / "nsi.parquet"
WAREHOUSE = REPO / "test" / "warehouse-notebook"
OUT_DIR = REPO / "test" / "exports-notebook"
CATALOG = "rqm"

print(f"repo:      {REPO}")
print(f"parquet:   {PARQUET}  ({PARQUET.stat().st_size / 1024 / 1024:.1f} MB)")
print(f"warehouse: {WAREHOUSE}")
print(f"exports:   {OUT_DIR}")

repo:      /workspaces/rqm-data-model
parquet:   /workspaces/rqm-data-model/test/data/nsi.parquet  (5.1 MB)
warehouse: /workspaces/rqm-data-model/test/warehouse-notebook
exports:   /workspaces/rqm-data-model/test/exports-notebook


## 1. Boot Spark + Sedona + Iceberg

`tools/ingest_nsi.py` already encapsulates the session configuration
(Sedona + Iceberg packages, Kryo serializer, local Hadoop catalog). We
reuse it here so the notebook and the CLI stay in sync.

In [ ]:
import shutil

if WAREHOUSE.exists():
    shutil.rmtree(WAREHOUSE)
WAREHOUSE.mkdir(parents=True, exist_ok=True)

from ingest_nsi import (
    NSI_SCALAR_FIELDS,
    SparkSettings,
    apply_schema,
    build_spark,
    load_buildings,
    load_geometries,
    sedona_geom_sql,
)

settings = SparkSettings(
    warehouse=WAREHOUSE,
    catalog=CATALOG,
    master="local[*]",  # Use all available cores locally
    conf={"spark.driver.memory": "8g"},
)  # Set driver memory to 8 GB
spark = build_spark(settings)

:: loading settings :: url = jar:file:/opt/conda/envs/rqm-base/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/vscode/.ivy2.5.2/cache
The jars for the packages stored in: /home/vscode/.ivy2.5.2/jars
org.apache.sedona#sedona-spark-shaded-4.1_2.13 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
org.apache.iceberg#iceberg-spark-runtime-4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d11d4be0-7152-4401-91f7-9f5f7d7b1fbd;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-shaded-4.1_2.13;1.9.1 in central
	found org.datasyslab#geotools-wrapper;1.9.1-33.5 in central
	found org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.12.0 in central
:: resolution report :: resolve 118ms :: artifacts dl 5ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.12.0 from central in [default]
	org.apache.sedona#sedona-

## 2. Peek at the raw NSI parquet

Before touching the schema, show what the source actually carries — field
names, row count, a handful of sample rows.

In [3]:
import pyarrow.parquet as pq

pf = pq.ParquetFile(str(PARQUET))
print(f"rows: {pf.metadata.num_rows:,}")
print(f"cols: {len(pf.schema_arrow)}")
print("columns:", sorted(f.name for f in pf.schema_arrow))

rows: 45,622
cols: 46
columns: ['bid', 'bldgtype', 'bldheight', 'cbfips', 'creprcnt', 'crerank', 'depindex', 'fd_id', 'firmzone', 'found_ht', 'found_type', 'ftprntid', 'ftprntsqft', 'ftprntsrc', 'fullrep', 'grnd_elv_m', 'ground_elv', 'med_yr_blt', 'novehprob', 'num_story', 'o65disable', 'occtype', 'ornl_hgh', 'ornl_low', 'ornl_med', 'pctlowclr', 'pop2amo65', 'pop2amu65', 'pop2pmo65', 'pop2pmu65', 'resunits', 'shape', 'source', 'sqft', 'st_damcat', 'static_bfe', 'students', 'u65disable', 'usastrucid', 'val_cont', 'val_struct', 'val_vehic', 'vehperunit', 'x', 'y', 'zone_sub']


In [4]:
# Preview a few rows (drop the WKB `shape` column for readability).
raw = spark.read.parquet(str(PARQUET))
raw.drop("shape").limit(5).toPandas()

26/11/09 22:17:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,fd_id,bid,usastrucid,occtype,st_damcat,resunits,cbfips,pop2amu65,pop2amo65,pop2pmu65,...,ornl_low,ornl_med,ornl_hgh,creprcnt,crerank,firmzone,zone_sub,static_bfe,ground_elv,grnd_elv_m
0,120519434,8645XMJC+QG9-2-2-3-1,,RES1-1SNB,RES,1,480850312021050,2,1,1,...,0.0,0.0,0.0,0.11,0.11,,,0.0,554.893677,169.131592
1,120519428,8645XMJC+QR7-3-3-3-3,,RES1-1SNB,RES,1,480850312021050,2,1,1,...,0.0,0.0,0.0,0.11,0.11,X,AREA OF MINIMAL FLOOD HAZARD,-9999.0,555.298950,169.255112
2,120519433,8645XMJC+QW2-1-3-2-2,,RES1-1SNB,RES,1,480850312021050,2,1,1,...,0.0,0.0,0.0,0.11,0.11,,,0.0,555.622192,169.353638
3,120519427,8645XMJF+Q22-2-2-2-3,,RES1-1SNB,RES,1,480850312021050,2,0,0,...,0.0,0.0,0.0,0.11,0.11,,,0.0,556.723755,169.689392
4,120519431,8645XMJF+Q54-2-3-2-2,,RES1-1SNB,RES,1,480850312021050,2,0,0,...,0.0,0.0,0.0,0.11,0.11,,,0.0,558.245422,170.153198


## 3. Create the Iceberg tables and ingest

`apply_schema` runs `tools/schema.sql` (idempotent `CREATE TABLE IF NOT
EXISTS`). `load_buildings` projects every NSI column into `rqm.buildings`
with a surrogate `building_id`; `load_geometries` derives one computation
point per structure (`role='point'`, `purpose='computation'`,
`is_selected_for_export=TRUE`, geom as WKB).

In [5]:
apply_schema(spark, CATALOG)
n_b = load_buildings(spark, PARQUET, CATALOG)
n_g = load_geometries(spark, PARQUET, CATALOG)
print(f"loaded: {n_b:,} buildings, {n_g:,} geometries")

loaded: 45,622 buildings, 45,622 geometries


## 4. Validate the ingest

Assert the ADR-nsi-minimal invariants directly — the same checks
`test/test_nsi_ingest.py` runs in CI, inline here so a stakeholder
flipping through the notebook can see them pass.

In [6]:
# Row-count parity
assert spark.table(f"{CATALOG}.buildings").count() == pf.metadata.num_rows

# Field-coverage / release-drift guard
parquet_cols = {f.name for f in pf.schema_arrow}
unmapped = parquet_cols - set(NSI_SCALAR_FIELDS) - {"shape"}
assert not unmapped, f"unmapped NSI columns: {unmapped}"

# Every building has its computation-selected point
missing = spark.sql(f"""
    SELECT COUNT(*) AS n
    FROM {CATALOG}.buildings b
    LEFT JOIN {CATALOG}.geometries g
      ON g.building_id = b.building_id
     AND g.role = 'point'
     AND g.purpose = 'computation'
     AND g.is_selected_for_export = TRUE
    WHERE g.geometry_id IS NULL
""").collect()[0]["n"]
assert missing == 0, f"{missing} buildings are missing a computation point"

# Geometry validity + CRS
invalid = spark.sql(f"""
    SELECT COUNT(*) AS n
    FROM {CATALOG}.geometries
    WHERE NOT ST_IsValid({sedona_geom_sql()})
       OR spark_st_srid(geom) != 3857 OR crs != 'EPSG:3857'
""").collect()[0]["n"]
assert invalid == 0, f"{invalid} geometries failed validity / CRS"

# Exactly one selected point per (building, purpose=computation)
dupes = spark.sql(f"""
    SELECT building_id, COUNT(*) AS n
    FROM {CATALOG}.geometries
    WHERE purpose = 'computation' AND is_selected_for_export = TRUE
    GROUP BY building_id
    HAVING COUNT(*) != 1
""").count()
assert dupes == 0, f"{dupes} buildings violate the selected-geometry invariant"

print("All ADR-nsi-minimal validation rules passed.")

All ADR-nsi-minimal validation rules passed.


## 5. Example analytical queries

A small, representative sample of what the ingested schema answers
directly. None of these should look surprising to anyone who has worked
with NSI before — the point is that they are one SQL hop from the
canonical table.

### 5.1 — Structures by occupancy type

In [7]:
spark.sql(f"""
    SELECT occtype, COUNT(*) AS n_structures
    FROM {CATALOG}.buildings
    GROUP BY occtype
    ORDER BY n_structures DESC
    LIMIT 10
""").toPandas()

,occtype,n_structures
0,RES1-1SNB,23223
1,RES1-2SNB,14634
2,AGR1,1923
3,COM1,1645
4,RES2,1045
5,RES3A,776
6,RES1-1SWB,491
7,RES1-2SWB,304
8,COM8,282
9,COM2,236


### 5.2 — Total exposed value by census block (top 10)

In [8]:
spark.sql(f"""
    SELECT cbfips,
           COUNT(*)                     AS n_structures,
           ROUND(SUM(val_struct), 0)    AS structure_value_usd,
           ROUND(SUM(val_cont), 0)      AS contents_value_usd,
           ROUND(SUM(val_struct + val_cont), 0) AS total_value_usd
    FROM {CATALOG}.buildings
    WHERE cbfips IS NOT NULL
    GROUP BY cbfips
    ORDER BY total_value_usd DESC
    LIMIT 10
""").toPandas()

,cbfips,n_structures,structure_value_usd,contents_value_usd,total_value_usd
0,483970403041006,146,400301624.0,321147543.0,721449167.0
1,483970404072005,581,413984349.0,210029502.0,624013851.0
2,483970404082001,547,353878676.0,224015697.0,577894373.0
3,483970405091003,481,366787648.0,183990192.0,550777840.0
4,483970403052062,75,260169695.0,287222050.0,547391745.0
5,483970405102001,299,308629631.0,234670982.0,543300613.0
6,483970404121001,622,322467184.0,164717579.0,487184763.0
7,483970403051001,135,198204628.0,244267697.0,442472324.0
8,483970405072016,102,269423654.0,171958232.0,441381886.0
9,483970405132001,263,256196865.0,149663741.0,405860606.0


### 5.3 — Daytime vs nighttime population, SFHA vs non-SFHA

Rough Special Flood Hazard Area classification by FIRM zone letter.

In [9]:
spark.sql(f"""
    SELECT
        CASE WHEN firmzone IN ('A','AE','AO','AH','V','VE') THEN 'SFHA' ELSE 'Non-SFHA' END AS zone_class,
        COUNT(*)                                                 AS structures,
        ROUND(SUM(pop2amu65 + pop2amo65), 0)                     AS pop_2am,
        ROUND(SUM(pop2pmu65 + pop2pmo65), 0)                     AS pop_2pm,
        ROUND(SUM(u65disable + o65disable), 0)                   AS disabled_population
    FROM {CATALOG}.buildings
    GROUP BY zone_class
    ORDER BY structures DESC
""").toPandas()

,zone_class,structures,pop_2am,pop_2pm,disabled_population
0,Non-SFHA,45114,135921.0,106051.0,8667.0
1,SFHA,508,1424.0,1056.0,132.0


### 5.4 — Structures below the base flood elevation

Comparing ground elevation to the static BFE gives a first-cut look at
which structures sit below the regulatory flood surface. Depth here is
`static_bfe - ground_elv` in feet; positive means the ground is below
the BFE.

In [10]:
spark.sql(f"""
    SELECT
        firmzone,
        COUNT(*)                                                 AS n_structures,
        SUM(CASE WHEN ground_elv < static_bfe THEN 1 ELSE 0 END) AS below_bfe,
        ROUND(AVG(static_bfe - ground_elv), 2)                   AS avg_depth_below_bfe_ft
    FROM {CATALOG}.buildings
    WHERE static_bfe IS NOT NULL AND ground_elv IS NOT NULL
    GROUP BY firmzone
    ORDER BY below_bfe DESC NULLS LAST
    LIMIT 10
""").toPandas()

,firmzone,n_structures,below_bfe,avg_depth_below_bfe_ft
0,AE,108,0,-10494.06
1,A,400,0,-10525.34
2,X,45055,0,-10534.32
3,,59,0,-490.96


## 6. Export an analysis-ready flat table

Join `buildings` with its computation-selected geometry, reproject the
point from EPSG:3857 to WGS84 (EPSG:4326), and select the fields a
downstream flood-loss engine typically consumes: identity, occupancy /
structural descriptors, value components, regulatory elevation, and
day/night population exposure.

Output format: single-part parquet (primary analysis format) and CSV
(for inspection in spreadsheets). Spark writes each as a directory
containing one `part-*.ext` file, because `.coalesce(1)` is in effect.

In [11]:
flat = spark.sql(f"""
    WITH pt AS (
        SELECT building_id,
               ST_Transform({sedona_geom_sql()}, 'EPSG:3857', 'EPSG:4326') AS geom_wgs84
        FROM {CATALOG}.geometries
        WHERE role = 'point' AND purpose = 'computation' AND is_selected_for_export = TRUE
    )
    SELECT
        b.building_id, b.fd_id, b.cbfips,
        b.occtype, b.st_damcat, b.bldgtype,
        b.num_story, b.sqft, b.bldheight,
        b.found_type, b.found_ht,
        b.val_struct, b.val_cont, b.val_vehic,
        b.firmzone, b.static_bfe, b.ground_elv,
        b.pop2amu65 + b.pop2amo65 AS pop_2am,
        b.pop2pmu65 + b.pop2pmo65 AS pop_2pm,
        b.u65disable + b.o65disable AS pop_disabled,
        ST_X(pt.geom_wgs84) AS lon_wgs84,
        ST_Y(pt.geom_wgs84) AS lat_wgs84
    FROM {CATALOG}.buildings b
    JOIN pt USING (building_id)
""")
flat.cache()
print(f"flat table: {flat.count():,} rows x {len(flat.columns)} cols")
flat.limit(5).toPandas()

flat table: 45,622 rows x 22 cols


,building_id,fd_id,cbfips,occtype,st_damcat,bldgtype,num_story,sqft,bldheight,found_type,...,val_cont,val_vehic,firmzone,static_bfe,ground_elv,pop_2am,pop_2pm,pop_disabled,lon_wgs84,lat_wgs84
0,120262583,120262583,,RES2,RES,H,1,1248.0,4.09,C,...,78326.074751,0.0,,0.0,561.216309,0.0,0.0,0.0,-96.392259,32.982103
1,120262564,120262564,,RES3A,RES,M,1,1677.0,2.54,S,...,101128.422569,0.0,,0.0,555.732788,0.0,0.0,0.0,-96.395190,32.982106
2,127618226,127618226,,RES2,RES,H,1,1408.0,4.29,C,...,87246.720000,0.0,,0.0,495.375275,0.0,0.0,0.0,-96.349950,32.813939
3,125867488,125867488,,RES1-1SNB,RES,M,1,3000.0,0.00,S,...,155897.259056,0.0,,0.0,507.978668,0.0,0.0,0.0,-96.470410,32.814281
4,125867494,125867494,,RES1-2SNB,RES,M,2,3545.0,0.00,S,...,184218.598197,0.0,,0.0,509.282562,0.0,0.0,0.0,-96.468645,32.814367


In [12]:
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

parquet_dir = OUT_DIR / "nsi_analysis_ready.parquet"
csv_dir = OUT_DIR / "nsi_analysis_ready.csv"

flat.coalesce(1).write.mode("overwrite").parquet(str(parquet_dir))
flat.coalesce(1).write.mode("overwrite").option("header", True).csv(str(csv_dir))

for d in (parquet_dir, csv_dir):
    for f in sorted(d.glob("part-*")):
        print(f"wrote: {f}  ({f.stat().st_size / 1024:.1f} KB)")

wrote: /workspaces/rqm-data-model/test/exports-notebook/nsi_analysis_ready.parquet/part-00000-4f480a61-f388-45cd-8c6f-d85ddae47f62-c000.snappy.parquet  (2255.5 KB)
wrote: /workspaces/rqm-data-model/test/exports-notebook/nsi_analysis_ready.csv/part-00000-0492a6b0-dddb-48be-957e-49998fb6a61b-c000.csv  (8291.2 KB)


### 6.1 — Read the exported parquet back as pandas

In [13]:
import pandas as pd

part = sorted((OUT_DIR / "nsi_analysis_ready.parquet").glob("part-*.parquet"))[0]
flat_pd = pd.read_parquet(part)
print(f"{len(flat_pd):,} rows, {len(flat_pd.columns)} cols")
flat_pd.head()

45,622 rows, 22 cols


,building_id,fd_id,cbfips,occtype,st_damcat,bldgtype,num_story,sqft,bldheight,found_type,...,val_cont,val_vehic,firmzone,static_bfe,ground_elv,pop_2am,pop_2pm,pop_disabled,lon_wgs84,lat_wgs84
0,120262583,120262583,,RES2,RES,H,1,1248.0,4.09,C,...,78326.074751,0.0,,0.0,561.216309,0.0,0.0,0.0,-96.392259,32.982103
1,120262564,120262564,,RES3A,RES,M,1,1677.0,2.54,S,...,101128.422569,0.0,,0.0,555.732788,0.0,0.0,0.0,-96.395190,32.982106
2,127618226,127618226,,RES2,RES,H,1,1408.0,4.29,C,...,87246.720000,0.0,,0.0,495.375275,0.0,0.0,0.0,-96.349950,32.813939
3,125867488,125867488,,RES1-1SNB,RES,M,1,3000.0,0.00,S,...,155897.259056,0.0,,0.0,507.978668,0.0,0.0,0.0,-96.470410,32.814281
4,125867494,125867494,,RES1-2SNB,RES,M,2,3545.0,0.00,S,...,184218.598197,0.0,,0.0,509.282562,0.0,0.0,0.0,-96.468645,32.814367


## 7. Teardown

In [14]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
